In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').getOrCreate()

## PySpark DataFrame Broadcast Join

A broadcast join is an optimization technique in PySpark where a small DataFrame is sent to all worker nodes, avoiding expensive shuffle operations.

### WHEN TO USE BROADCAST JOIN
- One DataFrame is significantly smaller than the other (typically < 10MB by default)
- The small table can fit into memory on each executor
- Want to avoid costly shuffle operations across the network
- Works only for equi joins '=' (joins based on equality conditions)

### HOW IT WORKS INTERNALLY

1. DRIVER SIDE:
    - Collects the smaller DataFrame completely into the driver's memory
    - Serializes the data into a broadcast variable
    - Calculates the size and decides if broadcast is feasible

2. BROADCAST DISTRIBUTION:
    - Driver sends the serialized small table to all executors
    - Uses efficient peer-to-peer distribution (BitTorrent-like protocol) to broadcast data
    - Data is cached on each executor node

3. EXECUTOR SIDE:
    - Each executor deserializes the broadcast variable
    - Caches it in memory for quick lookup
    - Performs local hash join without shuffling large table

4. JOIN EXECUTION:
    - Each partition of the large table is processed independently
    - For each row, looks up matching keys in the broadcast hash table
    - No network shuffle required for the large dataset

### CONFIGURATION PARAMETERS

`spark.sql.autoBroadcastJoinThreshold`:
  - Default: 10MB (10485760 bytes)
  - Set to -1 to disable automatic broadcast
  - Catalyst optimizer uses this to decide join strategy

`spark.sql.broadcastTimeout`:
  - Default: 300 seconds
  - Time to wait for broadcast to complete

In [19]:
large_df = spark.range(1000000).toDF("id")
large_df = large_df.withColumn("value", large_df.id * 2)
large_df.show(5)

+---+-----+
| id|value|
+---+-----+
|  0|    0|
|  1|    2|
|  2|    4|
|  3|    6|
|  4|    8|
+---+-----+
only showing top 5 rows


In [28]:
small_df = spark.range(100).toDF("id")
small_df = small_df.withColumn("emp_id", (small_df.id + 100).cast("string"))
small_df.show(5)

+---+------+
| id|emp_id|
+---+------+
|  0|   100|
|  1|   101|
|  2|   102|
|  3|   103|
|  4|   104|
+---+------+
only showing top 5 rows


In [37]:
# Check the broadcast threshold value
print(f" In Bytes: {spark.conf.get('spark.sql.autoBroadcastJoinThreshold')}")

print(f" In MB: {int(spark.conf.get('spark.sql.autoBroadcastJoinThreshold')[:-1])/1024/1024}")

 In Bytes: 10485760b
 In MB: 10.0


In [30]:
# Automatic Broadcast Join (if small_df size < threshold)
joined = large_df.join(small_df, "id")
joined.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [id#96L, emp_id#98, value#35L]
   +- BroadcastHashJoin [id#96L], [id#33L], Inner, BuildLeft, false
      :- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=159]
      :  +- Project [id#96L, cast((id#96L + 100) as string) AS emp_id#98]
      :     +- Range (0, 100, step=1, splits=2)
      +- Project [id#33L, (id#33L * 2) AS value#35L]
         +- Range (0, 1000000, step=1, splits=2)




In [38]:
# Disbale auto broadcast join
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', -1)
joined = large_df.join(small_df, "id")
joined.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [id#33L, value#35L, emp_id#98]
   +- SortMergeJoin [id#33L], [id#96L], Inner
      :- Sort [id#33L ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(id#33L, 200), ENSURE_REQUIREMENTS, [plan_id=214]
      :     +- Project [id#33L, (id#33L * 2) AS value#35L]
      :        +- Range (0, 1000000, step=1, splits=2)
      +- Sort [id#96L ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(id#96L, 200), ENSURE_REQUIREMENTS, [plan_id=215]
            +- Project [id#96L, cast((id#96L + 100) as string) AS emp_id#98]
               +- Range (0, 100, step=1, splits=2)




In [39]:
# Explicitly broadcast the small DataFrame
joined = large_df.join(small_df.hint("broadcast"), "id")
joined.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [id#33L, value#35L, emp_id#98]
   +- BroadcastHashJoin [id#33L], [id#96L], Inner, BuildRight, false
      :- Project [id#33L, (id#33L * 2) AS value#35L]
      :  +- Range (0, 1000000, step=1, splits=2)
      +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=244]
         +- Project [id#96L, cast((id#96L + 100) as string) AS emp_id#98]
            +- Range (0, 100, step=1, splits=2)


